# Subscription Service Game Prediction - Universal Model
## Two-Tier System + XGBoost + Log-Transform for Accuracy

**Compatible with:** Xbox Game Pass, PS Plus, Epic Games

**Key Changes from V6:**
- **Target Variable Transformation:** $\ln(days\_to\_service)$ used for training to handle the skewed data distribution (Step 5).
- **Improved XGBoost Regularization:** Added `reg_alpha` and `reg_lambda` to reduce overfitting and improve Test MAE (Step 6).

---
## 🎮 CONFIGURE YOUR PLATFORM HERE 👇

In [10]:
# ============================================================================
# 🔧 PLATFORM CONFIGURATION - CHANGE THESE VARIABLES
# ============================================================================

# Platform name (for display and file naming)
PLATFORM_NAME = "Xbox"  # Change to: "Xbox", "PSPlus", "Epic"
PLATFORM_DISPLAY = "Xbox Game Pass Ultimate"  # Change to: "Xbox Game Pass", "PS Plus Extra", "Epic Games"

# Input CSV file
INPUT_CSV = "Xbox.csv"  # Change to: "Xbox.csv", "PSPlus.csv", "Epic.csv"

# Column name in CSV for when game was added to service
DATE_COLUMN = "Added to Service"  # Usually: "Added to Service" or "Date Added"

# Date format in your CSV (adjust if needed)
DATE_FORMAT = "%m/%d/%Y"  # Common formats: "%m/%d/%Y", "%Y-%m-%d", "%d/%m/%Y"

# Average repeat interval for this platform (in months)
# Epic: 18-19 months, Xbox/PS Plus: 24+ months (repeats are rare)
AVG_REPEAT_INTERVAL = 42.3  # Change to: 18.9 for Epic, 24.0+ for Xbox/PS Plus

# Confidence adjustment for repeats (Epic games repeat more often)
# Higher for Epic (more predictable), lower for Xbox/PS (less common)
REPEAT_CONFIDENCE_MULTIPLIER = 0.75  # 1.0 for Epic, 0.75 for Xbox/PS

# ============================================================================
# 🎯 MODEL QUALITY ADJUSTMENT (Based on Test R² Performance)
# ============================================================================
# Use this to adjust confidence based on actual model performance
# High R² (0.4-0.7) = Good predictions, use 1.0
# Medium R² (0.2-0.4) = Moderate predictions, use 0.7-0.8
# Low R² (0.0-0.2) = Poor predictions, use 0.5-0.6

MODEL_QUALITY_MULTIPLIER = 0.6  # Adjust based on Test R² from training
MAX_CONFIDENCE_CAP = 70  # Maximum confidence percentage (95 for Epic, 70 for poor models)

# Platform-specific disclaimer
PREDICTION_DISCLAIMER = "High uncertainty - PS Plus catalog patterns are unpredictable"

# ============================================================================

# Output file names (automatically generated based on platform)
OUTPUT_MODEL = f"xgb_{PLATFORM_NAME.lower()}_model.pkl"
OUTPUT_ENCODER = f"publisher_encoder_{PLATFORM_NAME.lower()}.pkl"
OUTPUT_STATS = f"publisher_statistics_{PLATFORM_NAME.lower()}.csv"

# ============================================================================
print(f"✓ Configuration loaded for {PLATFORM_DISPLAY}")
print(f"  Input: {INPUT_CSV}")
print(f"  Model Quality Multiplier: {MODEL_QUALITY_MULTIPLIER}")
print(f"  Max Confidence Cap: {MAX_CONFIDENCE_CAP}%")
print(f"  Output Model: {OUTPUT_MODEL}")
print(f"  Output Stats: {OUTPUT_STATS}")
print(f"  Output Encoder: {OUTPUT_ENCODER}")


✓ Configuration loaded for Xbox Game Pass Ultimate
  Input: Xbox.csv
  Model Quality Multiplier: 0.6
  Max Confidence Cap: 70%
  Output Model: xgb_xbox_model.pkl
  Output Stats: publisher_statistics_xbox.csv
  Output Encoder: publisher_encoder_xbox.pkl


In [11]:
# Step 1: Import Libraries
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle

print("✓ Libraries imported successfully!")

✓ Libraries imported successfully!


In [12]:
# Step 2: Load and Analyze Data
df = pd.read_csv(INPUT_CSV)
df_clean = df[df['game_name'].notna()].copy()

# Parse dates
def parse_dates(date_str):
    if pd.isna(date_str):
        return pd.NaT
    try:
        return pd.to_datetime(date_str, format=DATE_FORMAT, errors='coerce')
    except:
        return pd.NaT

df_clean['added_to_service'] = df_clean[DATE_COLUMN].apply(parse_dates)
df_clean['release_date'] = df_clean['release_date'].apply(parse_dates)

print(f"✓ Loaded {PLATFORM_DISPLAY} data")
print(f"Total games: {len(df_clean)}")
print(f"Unique games: {df_clean['game_name'].nunique()}")

# Analyze repeat patterns
game_appearances = df_clean['game_name'].value_counts()
repeat_games = game_appearances[game_appearances > 1]
print(f"\nGames with repeat appearances: {len(repeat_games)}")
print(f"Repeat rate: {len(repeat_games)/len(game_appearances)*100:.1f}%")

# Calculate repeat statistics
repeat_analysis = []
for game_name in repeat_games.index:
    game_dates = df_clean[df_clean['game_name'] == game_name]['added_to_service'].dropna().sort_values()
    if len(game_dates) >= 2:
        intervals = [(game_dates.iloc[i+1] - game_dates.iloc[i]).days for i in range(len(game_dates)-1)]
        repeat_analysis.append({'game': game_name, 'avg_interval_months': np.mean(intervals)/30})

repeat_df = pd.DataFrame(repeat_analysis)
if len(repeat_df) > 0:
    actual_avg_interval = repeat_df['avg_interval_months'].mean()
    print(f"Actual average repeat interval: {actual_avg_interval:.1f} months")
    print(f"Configured repeat interval: {AVG_REPEAT_INTERVAL:.1f} months")
    if abs(actual_avg_interval - AVG_REPEAT_INTERVAL) > 6:
        print(f"⚠️  WARNING: Consider updating AVG_REPEAT_INTERVAL to {actual_avg_interval:.1f}")
else:
    print("Using configured repeat interval (no repeat data available)")

✓ Loaded Xbox Game Pass Ultimate data
Total games: 2119
Unique games: 1972

Games with repeat appearances: 143
Repeat rate: 7.3%
Actual average repeat interval: 42.3 months
Configured repeat interval: 42.3 months


In [13]:
# Step 3: Prepare Training Data for XGBoost
df_clean['days_to_service'] = (df_clean['added_to_service'] - df_clean['release_date']).dt.days

# Extract primary publisher
df_clean['primary_publisher'] = df_clean['publisher'].apply(
    lambda x: str(x).split(',')[0].strip() if pd.notna(x) else 'Unknown'
)

# Filter valid training data
training_data = df_clean[
    (df_clean['days_to_service'].notna()) & 
    (df_clean['days_to_service'] >= 1) & # Ensure days_to_service > 0 for log transform
    (df_clean['primary_publisher'] != 'Unknown')
].copy()

print(f"Training samples: {len(training_data)}")
print(f"Unique publishers: {training_data['primary_publisher'].nunique()}")

# Fill missing Metacritic scores with median
median_metacritic = training_data['metacritic_score'].median()
training_data['metacritic_score'] = training_data['metacritic_score'].fillna(median_metacritic)

print(f"Median Metacritic score: {median_metacritic}")

Training samples: 1749
Unique publishers: 397
Median Metacritic score: 78.0


In [14]:
# Step 4: Feature Engineering
# Calculate publisher statistics
publisher_stats = training_data.groupby('primary_publisher').agg({
    'days_to_service': ['mean', 'median', 'std', 'count'],
    'metacritic_score': 'mean'
}).reset_index()

publisher_stats.columns = ['publisher', 'pub_avg_days', 'pub_median_days', 'pub_std_days', 'pub_count', 'pub_avg_meta']
publisher_stats['pub_cv'] = publisher_stats['pub_std_days'] / publisher_stats['pub_avg_days']
publisher_stats['pub_cv'] = publisher_stats['pub_cv'].fillna(0.5)  # Default CV for publishers with one game

# Merge publisher stats back to training data
training_data = training_data.merge(publisher_stats, left_on='primary_publisher', right_on='publisher', how='left')

# Encode publisher
le_publisher = LabelEncoder()
training_data['publisher_encoded'] = le_publisher.fit_transform(training_data['primary_publisher'])

# Calculate game age at service release
training_data['game_age_years'] = training_data['days_to_service'] / 365

print("✓ Features engineered")
print(f"Publishers encoded: {len(le_publisher.classes_)}")
print(f"\nTop 5 publishers by game count:")
print(publisher_stats.nlargest(5, 'pub_count')[['publisher', 'pub_count', 'pub_avg_days', 'pub_cv']])

# Save publisher stats and encoder
publisher_stats.to_csv(OUTPUT_STATS, index=False)
with open(OUTPUT_ENCODER, 'wb') as f:
    pickle.dump(le_publisher, f)

print(f"\n✓ Saved {OUTPUT_STATS}")
print(f"✓ Saved {OUTPUT_ENCODER}")

✓ Features engineered
Publishers encoded: 397

Top 5 publishers by game count:
                 publisher  pub_count  pub_avg_days    pub_cv
117        Electronic Arts        162   2713.808642  1.013361
211      Microsoft Studios        132   2643.545455  0.737063
358  Ubisoft Entertainment        100   2338.750000  0.634412
322            Square Enix         55   1728.545455  0.912493
296                   SEGA         50   2902.900000  1.044966

✓ Saved publisher_statistics_xbox.csv
✓ Saved publisher_encoder_xbox.pkl


In [15]:
# Step 5: Prepare XGBoost Features (Target Transformation Applied)
feature_cols = [
    'metacritic_score',
    'publisher_encoded',
    'pub_avg_days',
    'pub_count',
    'pub_cv'
]

X = training_data[feature_cols].copy()

# *** TWEAK: Apply Log Transformation to the Target Variable (y) ***
# This helps normalize the skewed time-to-service data and reduce MAE.
y = np.log(training_data['days_to_service'].copy())
y_original = training_data['days_to_service'].copy() # Keep for final metrics

# Handle any remaining NaNs
X = X.fillna(X.median())

# Split data
X_train, X_test, y_train_log, y_test_log, y_train_orig, y_test_orig = train_test_split(
    X, y, y_original, test_size=0.2, random_state=42
)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"\nTarget variable has been log-transformed for training.")
print(f"Features: {feature_cols}")

Training set: 1399 samples
Test set: 350 samples

Target variable has been log-transformed for training.
Features: ['metacritic_score', 'publisher_encoded', 'pub_avg_days', 'pub_count', 'pub_cv']


In [16]:
# Step 6: Train XGBoost Model (with Regularization)
print(f"Training XGBoost Regressor for {PLATFORM_DISPLAY}...\n")

# *** TWEAK: Added L1/L2 Regularization (reg_alpha, reg_lambda) to combat overfitting ***
xgb_model = xgb.XGBRegressor(
    n_estimators=300, # Increased estimators slightly for better fit after regularization
    learning_rate=0.05,
    max_depth=5, # Reduced max_depth from 6 to 5 to simplify model
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,  # L1 regularization
    reg_lambda=0.1, # L2 regularization
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train_log)

# Evaluate
# Predictions are in log scale; must be exponentiated for final metric calculation
y_pred_train_log = xgb_model.predict(X_train)
y_pred_test_log = xgb_model.predict(X_test)

y_pred_train_orig = np.exp(y_pred_train_log)
y_pred_test_orig = np.exp(y_pred_test_log)

print("=" * 60)
print(f"MODEL PERFORMANCE - {PLATFORM_DISPLAY} (Log-Transformed Target)")
print("=" * 60)
print(f"Train MAE: {mean_absolute_error(y_train_orig, y_pred_train_orig):.2f} days ({mean_absolute_error(y_train_orig, y_pred_train_orig)/30:.1f} months)")
print(f"Test MAE:  {mean_absolute_error(y_test_orig, y_pred_test_orig):.2f} days ({mean_absolute_error(y_test_orig, y_pred_test_orig)/30:.1f} months)")
print(f"Train R²:  {r2_score(y_train_orig, y_pred_train_orig):.3f}")
print(f"Test R²:   {r2_score(y_test_orig, y_pred_test_orig):.3f}")

# Feature importance
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n" + "=" * 60)
print("FEATURE IMPORTANCE")
print("=" * 60)
print(importance_df.to_string(index=False))

# Save model
with open(OUTPUT_MODEL, 'wb') as f:
    pickle.dump(xgb_model, f)

print(f"\n✓ Model saved as {OUTPUT_MODEL}")

Training XGBoost Regressor for Xbox Game Pass Ultimate...

MODEL PERFORMANCE - Xbox Game Pass Ultimate (Log-Transformed Target)
Train MAE: 846.60 days (28.2 months)
Test MAE:  1211.38 days (40.4 months)
Train R²:  0.751
Test R²:   0.555

FEATURE IMPORTANCE
          feature  importance
     pub_avg_days    0.617684
           pub_cv    0.104406
 metacritic_score    0.103573
        pub_count    0.094798
publisher_encoded    0.079539

✓ Model saved as xgb_xbox_model.pkl


In [17]:
# Step 7: Create Two-Tier Predictor Class (With Model Quality Adjustment)

class GameServicePredictor:
    def __init__(self, csv_path, xgb_model_path, publisher_stats_path, publisher_encoder_path, 
                 platform_name, avg_repeat_interval, repeat_confidence_mult, date_column, date_format,
                 model_quality_mult=1.0, max_confidence_cap=95, disclaimer=""):
        """Universal game service predictor with model quality adjustment"""
        self.platform_name = platform_name
        self.avg_repeat_interval = avg_repeat_interval
        self.repeat_confidence_mult = repeat_confidence_mult
        self.model_quality_mult = model_quality_mult  # NEW
        self.max_confidence_cap = max_confidence_cap  # NEW
        self.disclaimer = disclaimer  # NEW
        
        # Load data
        self.df = pd.read_csv(csv_path)
        self.df = self.df[self.df['game_name'].notna()].copy()
        self.df['added_to_service'] = pd.to_datetime(self.df[date_column], format=date_format, errors='coerce')
        self.df['release_date'] = pd.to_datetime(self.df['release_date'], format=date_format, errors='coerce')
        
        # Load models
        with open(xgb_model_path, 'rb') as f:
            self.xgb_model = pickle.load(f)
        with open(publisher_encoder_path, 'rb') as f:
            self.publisher_encoder = pickle.load(f)
        
        self.publisher_stats = pd.read_csv(publisher_stats_path)
        self.median_metacritic = 75
    
    def _calculate_confidence(self, sample_size, variance_coefficient=None, has_metacritic=False, is_repeat=False):
        """Calculate confidence score with model quality adjustment"""
        if is_repeat:
            if sample_size >= 3:
                base = 85
            elif sample_size == 2:
                base = 75
            else:
                base = 65
            base = int(base * self.repeat_confidence_mult)
        else:
            if sample_size >= 20:
                base = 80
            elif sample_size >= 10:
                base = 70
            elif sample_size >= 5:
                base = 60
            elif sample_size >= 3:
                base = 50
            else:
                base = 40
        
        if variance_coefficient is not None:
            if variance_coefficient < 0.3:
                base += 10
            elif variance_coefficient < 0.5:
                base += 5
            elif variance_coefficient > 0.8:
                base -= 10
        
        if has_metacritic:
            base += 5
        
        # Apply model quality multiplier (NEW)
        base = int(base * self.model_quality_mult)
        
        # Cap at maximum confidence (NEW)
        return max(min(int(base), self.max_confidence_cap), 5)
    
    def _months_to_bucket(self, months):
        """Convert predicted months to time bucket category"""
        if months <= 6:
            return 'within 6 months'
        elif months <= 12:
            return 'within 6-12 months'
        elif months <= 24:
            return 'more than 12 months'
        elif months <= 48:
            return 'more than 24 months'
        else:
            return 'as good as never (many years)'
    
    def check_if_appeared(self, game_name):
        """Check if game has appeared on service before"""
        appearances = self.df[self.df['game_name'].str.lower() == game_name.lower()]
        if len(appearances) == 0:
            return None
        
        dates = appearances['added_to_service'].dropna().sort_values()
        if len(dates) == 0:
            return {'appeared': True, 'repeat_count': len(appearances)}
        
        result = {
            'appeared': True,
            'repeat_count': len(dates),
            'last_appearance': dates.iloc[-1]
        }
        
        if len(dates) >= 2:
            intervals = [(dates.iloc[i+1] - dates.iloc[i]).days for i in range(len(dates)-1)]
            result['avg_interval_months'] = np.mean(intervals) / 30
            result['cv'] = np.std(intervals) / np.mean(intervals) if np.mean(intervals) > 0 else 0
        
        return result
    
    def predict_repeat(self, game_name):
        """TIER 1: Predict repeat appearance based on historical patterns"""
        history = self.check_if_appeared(game_name)
        if not history:
            return None
        
        months_since = (datetime.now() - history['last_appearance']).days / 30
        
        if history['repeat_count'] == 1:
            predicted_months = max(0, self.avg_repeat_interval - months_since)
            confidence = self._calculate_confidence(1, None, False, True)
            reasoning = f"Appeared once {months_since:.1f} months ago on {self.platform_name}. Avg repeat: ~{self.avg_repeat_interval:.0f} months."
        else:
            avg_interval = history['avg_interval_months']
            predicted_months = max(0, avg_interval - months_since)
            confidence = self._calculate_confidence(history['repeat_count'], history['cv'], False, True)
            reasoning = f"Appeared {history['repeat_count']} times on {self.platform_name}. Avg interval: {avg_interval:.0f} months. {months_since:.1f} months since last."
        
        # Add disclaimer if set (NEW)
        if self.disclaimer:
            reasoning += f" Note: {self.disclaimer}"
        
        return {
            'category': self._months_to_bucket(predicted_months),
            'confidence': confidence,
            'predicted_months': predicted_months,
            'reasoning': reasoning,
            'sample_size': history['repeat_count']
        }
    
    def predict_new_xgb(self, game_name, publisher, metacritic_score=None):
        """TIER 2: Predict new game using XGBoost"""
        if publisher not in self.publisher_encoder.classes_:
            return {
                'category': 'unknown (no record of publisher in service)',
                'confidence': 0,
                'reasoning': f"Publisher '{publisher}' not found in {self.platform_name} training data."
            }
        
        pub_stats = self.publisher_stats[self.publisher_stats['publisher'] == publisher]
        if len(pub_stats) == 0:
            return {
                'category': 'unknown (no record of publisher in service)',
                'confidence': 0,
                'reasoning': f"No statistics for publisher '{publisher}' on {self.platform_name}."
            }
        
        pub_stats = pub_stats.iloc[0]
        meta_score = metacritic_score if metacritic_score else self.median_metacritic
        publisher_encoded = self.publisher_encoder.transform([publisher])[0]
        
        features = np.array([[
            meta_score,
            publisher_encoded,
            pub_stats['pub_avg_days'],
            pub_stats['pub_count'],
            pub_stats['pub_cv']
        ]])
        
        predicted_days = self.xgb_model.predict(features)[0]
        predicted_months = predicted_days / 30
        
        confidence = self._calculate_confidence(
            int(pub_stats['pub_count']),
            pub_stats['pub_cv'],
            metacritic_score is not None,
            False
        )
        
        category = self._months_to_bucket(predicted_months)
        reasoning = f"XGBoost prediction for {self.platform_name}: {predicted_days:.0f} days ({predicted_months:.0f} months). Publisher '{publisher}' has {int(pub_stats['pub_count'])} games on service."
        
        # Add disclaimer if set (NEW)
        if self.disclaimer:
            reasoning += f" Note: {self.disclaimer}"
        
        return {
            'category': category,
            'confidence': confidence,
            'predicted_months': predicted_months,
            'predicted_days': predicted_days,
            'reasoning': reasoning,
            'publisher_game_count': int(pub_stats['pub_count']),
            'publisher_consistency': pub_stats['pub_cv']
        }
    
    def predict(self, game_name, publisher=None, metacritic_score=None):
        """Main prediction method - uses two-tier system"""
        # TIER 1: Check for repeat pattern
        repeat_pred = self.predict_repeat(game_name)
        if repeat_pred:
            return {
                'game_name': game_name,
                'platform': self.platform_name,
                'tier': 'Historical Lookup (Repeat Pattern)',
                **repeat_pred
            }
        
        # TIER 2: XGBoost prediction for new games
        if not publisher:
            return {
                'game_name': game_name,
                'platform': self.platform_name,
                'tier': 'Unknown',
                'category': 'unknown (no record of publisher in service)',
                'confidence': 0,
                'reasoning': 'No publisher provided and no historical data available.'
            }
        
        new_pred = self.predict_new_xgb(game_name, publisher, metacritic_score)
        return {
            'game_name': game_name,
            'publisher': publisher,
            'platform': self.platform_name,
            'tier': 'XGBoost ML Prediction (New Game)',
            **new_pred
        }

# Initialize predictor with quality adjustments
predictor = GameServicePredictor(
    csv_path=INPUT_CSV,
    xgb_model_path=OUTPUT_MODEL,
    publisher_stats_path=OUTPUT_STATS,
    publisher_encoder_path=OUTPUT_ENCODER,
    platform_name=PLATFORM_DISPLAY,
    avg_repeat_interval=AVG_REPEAT_INTERVAL,
    repeat_confidence_mult=REPEAT_CONFIDENCE_MULTIPLIER,
    date_column=DATE_COLUMN,
    date_format=DATE_FORMAT,
    model_quality_mult=MODEL_QUALITY_MULTIPLIER,  # NEW
    max_confidence_cap=MAX_CONFIDENCE_CAP,  # NEW
    disclaimer=PREDICTION_DISCLAIMER  # NEW
)

print(f"✓ Two-Tier XGBoost Predictor initialized for {PLATFORM_DISPLAY}!")
print(f"  Model quality adjustment: {MODEL_QUALITY_MULTIPLIER}x")
print(f"  Max confidence capped at: {MAX_CONFIDENCE_CAP}%")


✓ Two-Tier XGBoost Predictor initialized for Xbox Game Pass Ultimate!
  Model quality adjustment: 0.6x
  Max confidence capped at: 70%


In [18]:
# Step 8: Test the System
print("=" * 80)
print(f"TESTING TWO-TIER SYSTEM - {PLATFORM_DISPLAY}")
print("=" * 80)

# TIER 1 Test
sample_games = df_clean['game_name'].value_counts().head(3).index.tolist()
if len(sample_games) > 0:
    print(f"\n[TIER 1 TEST] Repeat Prediction")
    print("-" * 80)
    test1 = predictor.predict(sample_games[0])
    print(f"Game: {test1['game_name']}")
    print(f"Platform: {test1['platform']}")
    print(f"Tier: {test1['tier']}")
    print(f"Category: {test1['category']}")
    print(f"Confidence: {test1['confidence']}%")
    print(f"Reasoning: {test1['reasoning']}")

# TIER 2 Test
top_publisher = publisher_stats.nlargest(1, 'pub_count')['publisher'].iloc[0]
print(f"\n[TIER 2 TEST] New Game Prediction")
print("-" * 80)
test2 = predictor.predict('Hypothetical New Game', top_publisher, 85)
print(f"Game: {test2['game_name']}")
print(f"Publisher: {test2['publisher']}")
print(f"Platform: {test2['platform']}")
print(f"Tier: {test2['tier']}")
print(f"Category: {test2['category']}")
print(f"Confidence: {test2['confidence']}%")
print(f"Predicted: {test2.get('predicted_months', 0):.1f} months")
print(f"Reasoning: {test2['reasoning']}")

# Unknown publisher test
print(f"\n[TIER 2 TEST] Unknown Publisher")
print("-" * 80)
test3 = predictor.predict('Another Game', 'Unknown Publisher XYZ', 80)
print(f"Game: {test3['game_name']}")
print(f"Category: {test3['category']}")
print(f"Confidence: {test3['confidence']}%")
print(f"Reasoning: {test3['reasoning']}")

print("\n" + "=" * 80)
print(f"✓ {PLATFORM_DISPLAY} prediction system ready!")
print("=" * 80)

TESTING TWO-TIER SYSTEM - Xbox Game Pass Ultimate

[TIER 1 TEST] Repeat Prediction
--------------------------------------------------------------------------------
Game: Tom Clancy's The Division
Platform: Xbox Game Pass Ultimate
Tier: Historical Lookup (Repeat Pattern)
Category: more than 24 months
Confidence: 37%
Reasoning: Appeared 3 times on Xbox Game Pass Ultimate. Avg interval: 36 months. 1.0 months since last. Note: High uncertainty - PS Plus catalog patterns are unpredictable

[TIER 2 TEST] New Game Prediction
--------------------------------------------------------------------------------
Game: Hypothetical New Game
Publisher: Electronic Arts
Platform: Xbox Game Pass Ultimate
Tier: XGBoost ML Prediction (New Game)
Category: within 6 months
Confidence: 45%
Predicted: 0.3 months
Reasoning: XGBoost prediction for Xbox Game Pass Ultimate: 8 days (0 months). Publisher 'Electronic Arts' has 162 games on service. Note: High uncertainty - PS Plus catalog patterns are unpredictable

[T